<a href="https://colab.research.google.com/github/NomeCoder/Machine-fault-detector/blob/main/MLModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATASET_PATH = "/content/drive/MyDrive/microdata/Data"

Mounted at /content/drive


In [ ]:
import os
import numpy as np
import pandas as pd
from scipy.fft import fft
import scipy.io as sio # Import scipy.io

X = []
y = []

BASE_PATH = "/content/drive/MyDrive/microdata/Data"

for fault_type in os.listdir(BASE_PATH):
    fault_path = os.path.join(BASE_PATH, fault_type)

    if not os.path.isdir(fault_path):
        continue

    # Label
    if "Ideal" in fault_type:
        label = "normal"
    elif "Wear" in fault_type:
        label = "wear"
    elif "Cracking" in fault_type:
        label = "crack"
    else:
        continue

    # Now inside axis folders
    for axis_folder in os.listdir(fault_path):
        axis_path = os.path.join(fault_path, axis_folder)

        if not os.path.isdir(axis_path):
            continue

        for file in os.listdir(axis_path):
            file_path = os.path.join(axis_path, file)

            signal = None # Initialize signal to None

            # Check if the file is a .mat file
            if file_path.endswith('.mat'):
                try:
                    mat_contents = sio.loadmat(file_path)

                    # Find the most likely data array in the .mat file
                    data_candidate_key = None
                    max_size = -1
                    for key, value in mat_contents.items():
                        # Exclude MATLAB's internal keys and ensure it's a numpy array
                        if not key.startswith('__') and isinstance(value, np.ndarray):
                            current_size = value.size
                            if current_size > max_size:
                                max_size = current_size
                                data_candidate_key = key

                    if data_candidate_key:
                        potential_signal = mat_contents[data_candidate_key]
                        # If it's a 2D array and original code used df[1], take the second column
                        if potential_signal.ndim == 2 and potential_signal.shape[1] > 1:
                            signal = potential_signal[:, 1]
                        else:
                            # Otherwise, assume it's already the signal (possibly 1D or single column)
                            signal = potential_signal.flatten() # Ensure 1D
                    else:
                        print(f"Warning: No suitable array data found in {file_path}. Skipping.")
                        continue # Skip to the next file if no signal found

                except Exception as e:
                    print(f"Error loading .mat file {file_path}: {e}")
                    continue # Skip to the next file on error
            else:
                # Original logic for other file types (e.g., CSV, TXT)
                with open(file_path, "r", encoding="latin1") as f:
                    lines = f.readlines()

                start_index = None
                for idx, line in enumerate(lines):
                    if line.strip() and line.strip()[0].isdigit():
                        start_index = idx
                        break

                if start_index is None:
                    print(f"Warning: Could not determine start_index for {file_path}. Skipping.")
                    continue

                try:
                    df = pd.read_csv(file_path,
                                     encoding="latin1",
                                     skiprows=start_index,
                                     header=None)
                    signal = df[1].values
                except Exception as e:
                    print(f"Error reading non-.mat file {file_path}: {e}")
                    continue


            # Windowing
            WINDOW_SIZE = 1024
            OVERLAP = 512

            # Ensure signal is numeric and not empty before windowing
            if signal is None or not isinstance(signal, np.ndarray) or signal.size == 0:
                print(f"Warning: Signal is empty or not an array for {file_path}. Skipping feature extraction.")
                continue

            for start in range(0, len(signal) - WINDOW_SIZE, OVERLAP):
                window = signal[start:start+WINDOW_SIZE]

                # Ensure window has enough data
                if len(window) < WINDOW_SIZE:
                    continue # Skip if window is too small

                # ---- Feature extraction ----
                features = [
                    np.mean(window),
                    np.std(window),
                    np.sqrt(np.mean(window**2)),
                    np.max(window),
                    np.min(window),
                    pd.Series(window).kurtosis(),
                    pd.Series(window).skew(),
                    np.mean(np.abs(fft(window))[:WINDOW_SIZE//2]),
                    np.max(np.abs(fft(window))[:WINDOW_SIZE//2]),
                ]

                X.append(features)
                y.append(label)

X = np.array(X)
y = np.array(y)

print("Total Samples:", len(X))

Total Samples: 675


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

model = RandomForestClassifier(n_estimators=200)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", model.score(X_test, y_test))
print(classification_report(y_test, y_pred))

Accuracy: 0.8740740740740741
              precision    recall  f1-score   support

       crack       0.83      0.88      0.86        34
      normal       0.87      0.94      0.91        36
        wear       0.90      0.83      0.86        65

    accuracy                           0.87       135
   macro avg       0.87      0.89      0.88       135
weighted avg       0.88      0.87      0.87       135



In [ ]:
import joblib
joblib.dump(scaler, "scaler.pkl")

['scaler.pkl']